# Part 1: Say hi to the brain

Everything today runs on your laptop. No API keys, no cloud. By the end you can
turn your wifi off and it keeps working.

**How notebooks work:** click a grey cell, press **Shift+Enter** to run it.
Change anything in a cell and run it again. You can't break anything that a
kernel restart (the ⟳ button) won't fix.

### Step 0: get a brain

Pick the model that fits your laptop and run the next two cells. The first one
downloads it (once, with a progress bar). The second starts `llama-server` in the
background. Both are safe to re-run.

| `MODEL` | For | Download |
|---|---|---|
| `"gemma-4-E2B"` | most laptops (16 GB RAM, Apple Silicon, or a GPU) | ~3-4 GB |
| `"smolvlm2-2.2B"` | 8 GB RAM or older machines | ~1.7 GB |
| `"potato"` | anything. Tiny, fast, a bit dim | ~0.3 GB |
| `"gemma-4-E4B"` | big machines that want smarter | ~5-6 GB |
| `"qwen2.5-omni-3B"` | the bonus notebook 5: sees *and* hears (Gemma 4 does too) | ~3.6 GB |

You need llama.cpp installed. Not yet? `INSTALL.md` (in the file panel on the left)
has a guide for macOS, Windows and Linux.
Got the model on a USB stick? Copy its folder into `models/` and the download cell skips the internet.

In [ ]:
from agent import server

server.check()             # is llama.cpp installed? prints the version, or how to install it

MODEL = "gemma-4-E2B"      # pick a row from the table above

model_path, mmproj_path = server.download(MODEL)

In [ ]:
server.start(model_path, mmproj_path)   # starts it, or restarts it with the settings above. Safe to re-run.

The server keeps running in the background while you move between notebooks, even if
you restart a kernel. Useful later:

- to swap models, change `MODEL` and re-run both cells (start restarts the server for you)
- `server.tail()` shows the last lines of the server log when something goes wrong
- started `llama-server` yourself in a terminal instead? That works too, skip the two cells above

In [ ]:
# Run me first (Shift+Enter). Re-run me if you restart the kernel.
%load_ext autoreload
%autoreload 2

import config
from agent import nb, ears, mouth
from agent.llm import LocalLLM, system, user, assistant

llm = LocalLLM()
print("brain:", llm.model_name() if llm.health() else f"NOT RUNNING at {config.SERVER_URL} (run the Step 0 cells in 1_setup.ipynb)")

## Your first local answer

Three knobs. Change them, run the cell again.

In [ ]:
SYSTEM_PROMPT = "You are a helpful assistant running entirely on a laptop. Keep answers short."
TEMPERATURE = 0.7      # 0 = same answer every time, 1.5+ = chaos
MAX_TOKENS = 200       # try 10 and watch it stop mid-sentence

question = "In one sentence: what are you, and where are you running right now?"

reply = llm.chat([system(SYSTEM_PROMPT), user(question)],
                 on_token=nb.stream, temperature=TEMPERATURE, max_tokens=MAX_TOKENS)
print("\n\n", reply.stats())

## What actually went over the wire

That was one HTTP POST to `localhost:8080`. Here's the whole thing.
There's no SDK and nothing else to it. `agent/llm.py` is ~170 lines, mostly comments.

In [ ]:
import json
payload = {"messages": [system(SYSTEM_PROMPT), user(question)],
           "temperature": TEMPERATURE, "max_tokens": MAX_TOKENS, "stream": True}
print(json.dumps(payload, indent=2))

## A conversation

The model remembers nothing between calls. "Memory" is just us re-sending the
whole conversation every time. Run `chat(...)` a few times, then look at `history`.

In [ ]:
history = [system(SYSTEM_PROMPT)]

def chat(message):
    history.append(user(message))
    r = llm.chat(history, on_token=nb.stream, temperature=TEMPERATURE, max_tokens=MAX_TOKENS)
    history.append(assistant(r.text))
    print("\n\n", r.stats())

chat("My name is Sam and I like tacos.")

In [ ]:
chat("What's my name, and what should I have for lunch?")

In [ ]:
history   # this list is the model's entire memory. Try deleting an item and asking again.

## Can it see?

A quick test card: a red square and a blue circle. If your model gets this right,
vision is working.

In [ ]:
import cv2, numpy as np

card = np.full((300, 400, 3), 255, np.uint8)
cv2.rectangle(card, (40, 60), (180, 240), (0, 0, 220), -1)   # red (OpenCV is BGR)
cv2.circle(card, (290, 150), 70, (220, 80, 0), -1)           # blue
nb.show(card, width=300)

from agent.eyes import preprocess
r = llm.chat([user("What two shapes are in this image, and what colour is each?", images=[preprocess(card)])],
             on_token=nb.stream, max_tokens=60, temperature=0)

## Check your camera, mic and voice

Each cell is independent. If one fails, read the message, fix it or skip it:
everything later has a no-camera and a no-mic option.

In [ ]:
# CAMERA: 0 is your default webcam. Wrong one? try 1. No camera? put a photo path here.
from agent.eyes import Eyes
eyes = Eyes(config.CAMERA)
nb.show(eyes.frame())
eyes.close()

In [ ]:
# MIC + WHISPER: say "testing one two three". First run downloads whisper (~145 MB).
audio = nb.record(3)
nb.play(audio)
text, secs = ears.transcribe(audio)
print(f"whisper heard: {text!r} in {secs:.2f}s")

In [ ]:
# VOICE
mouth.speak("All systems go. No cloud required.")

## Try this

1. Open http://127.0.0.1:8080 in a browser. That's llama-server's own chat page. Drag a photo in.
   (Getting `"File Not Found"`? Some llama.cpp builds ship without the chat page. Nothing here needs it; skip this one.)
2. **Turn your wifi off.** Re-run the first answer cell. Still works?
3. `TEMPERATURE = 0`, run twice. Then `1.8`. When does it get weird?
4. `chat("What's today's date?")`. Why is it wrong? (Nothing reaches the model unless you put it in the prompt. Remember this for Part 4.)
5. Change `SYSTEM_PROMPT` to a pirate. Or "only answer in haiku".